In [ ]:
# !pip install gensim
# !pip install sumy
# !pip install spacy
# !python -m spacy download en_core_web_sm
# !pip install openai
# !pip install PyPDF2
# !pip install nltk
# !pip install summarizer
# !pip install bert-extractive-summarizer
# !pip install transformers
# nltk.download('punkt')

In [ ]:
#nltk.download('punkt')

In [1]:
# Example usage:
long_text = """

Electronic component: Focus to increase local value addition: We hosted Vinod Sharma, Managing Director, Deki Electronics, to understand  the key enablers that can help in developing an electronic component  ecosystem in India. Key takeaways—(1) Component PLI scheme will be vastly  different from Mobile PLI scheme with focus on value addition and capex  subsidies, (2) collaboration with global players will be the key for components  ecosystem development, (3) white goods, defense and railway will be key  drivers of component growth and (4) exports could be a long-term opportunity.

Component PLI scheme to be vastly different from Mobile PLI:  A component PLI remains a key ask of the Indian electronic industry. However,  given the lower asset turn (2X is a typical component asset turn) and higher  value addition (40%) versus mobile assembly business (20X asset turn and  ~10% value addition), Mr Sharma expects the contours of the Component PLI  to be very different from the Mobile PLI scheme. (1) Focus on value addition,  (2) capex-driven subsidy similar to semiconductor industry (versus production  incentive in case of mobile), (3) long duration (6 to 8 years duration of PLI  scheme given the longer-term lead time required to get a component factory  running) remain some of the key ask of component electronic industry players 

Partnerships and JVs with global majors the key across most segments:  Given limited domestic capabilities, manufacturing tie-ups with global leaders  through JVs will be the key for Indian component ecosystem to develop. Here, (1) traditional EMS players may look at backward integration for certain  components or (2) Indian components players could tie up with global peers.  Focus would be on tie-up with firms from Japan, Taiwan, South Korea, US and  Europe, with selective tie-ups with Chinese component companies.


White goods, defense and industrial first to be localized:  As per Mr Sharma, moving the entire mobile component value chain to India  would take the longest time given it is the most complex value chain. Among  consumer segments, he expects white goods as the easiest segment where a  local component ecosystem can be developed. Among industrial segment, defense driven by national security concerns, railways and EVs would be  segments where domestic component ecosystem would be first developed.

Exports a long-term opportunity:  Once players of scale have developed for a few of the components, then over time exports could be an area of growth that develops. Emergence of a  completely integrated player that does EMS as well as components seems  highly unlikely, according to Mr Sharma, given the very different margin, asset  turn and value addition that a component player has versus an EMS player.  Component players would be more product-focused and would look at global  markets to scale up, hence exports become a longer-term driver of growth. 


"""

In [2]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
import spacy

# Load the spaCy English model
nlp = spacy.load("en_core_web_sm")

def generate_summary(long_text, sentences_count_ratio=0.2, min_sentences=3):
    """
    Generate a summary of a long text using sumy's LsaSummarizer.
    
    Args:
    - long_text (str): The long text to be summarized.
    - sentences_count_ratio (float, optional): The ratio of sentences in the summary compared to the original text. 
                                               Defaults to 0.2 (20% of the original text).
    - min_sentences (int, optional): The minimum number of sentences in the summary. 
                                     Defaults to 3.
                                
    Returns:
    - summary (str): The summarized text.
    """
    total_sentences = long_text.count('.') + long_text.count('!') + long_text.count('?')
    sentences_count = max(min_sentences, int(total_sentences * sentences_count_ratio))
    
    parser = PlaintextParser.from_string(long_text, Tokenizer("english"))
    summarizer = LsaSummarizer()
    summary = summarizer(parser.document, sentences_count)
    summary_text = " ".join([str(sentence) for sentence in summary])
    return summary_text


summary = generate_summary(long_text, sentences_count_ratio=0.3, min_sentences=5)
print(summary)

Electronic component: Focus to increase local value addition: We hosted Vinod Sharma, Managing Director, Deki Electronics, to understand  the key enablers that can help in developing an electronic component  ecosystem in India. Key takeaways—(1) Component PLI scheme will be vastly  different from Mobile PLI scheme with focus on value addition and capex  subsidies, (2) collaboration with global players will be the key for components  ecosystem development, (3) white goods, defense and railway will be key  drivers of component growth and (4) exports could be a long-term opportunity. Partnerships and JVs with global majors the key across most segments:  Given limited domestic capabilities, manufacturing tie-ups with global leaders  through JVs will be the key for Indian component ecosystem to develop. Among  consumer segments, he expects white goods as the easiest segment where a  local component ecosystem can be developed. Emergence of a  completely integrated player that does EMS as wel

In [3]:
def answer_question(long_text, query):
    # Process the long text
    doc = nlp(long_text)
    
    # Process the query
    query_doc = nlp(query)
    
    # Check if the query contains any relevant keywords for summary requests
    summary_keywords = ["summary", "summarize", "brief"]
    is_summary_request = any(token.text.lower() in summary_keywords for token in query_doc)
    
    # If the query is a request for a summary, generate the summary for the specified section
    if is_summary_request:
        section = get_section(query, long_text)
        if section:
            return generate_summary(section)
        else:
            # Attempt to extract the specified section from the query
            query_section = extract_section(query)
            if query_section:
                section = get_section(query_section, long_text)
                if section:
                    return generate_summary(section)
                else:
                    return "Cannot find the specified section in the long text"
            else:
                return "Cannot understand the query. Please specify a section or topic for the summary."
    
    # Look for relevant information based on the context of the query
    relevant_info = []
    for sentence in doc.sents:
        if query.lower() in sentence.text.lower():
            relevant_info.append(sentence.text)
    
    # Process the relevant information and generate a response
    if relevant_info:
        return "\n".join(relevant_info)
    else:
        return "Cannot provide a response based on long_text"

def get_section(query, long_text):
    # Split the long text into sections based on the section headers
    sections = long_text.split("\n\n")
    
    # Search for the section that matches the query
    for section in sections:
        if query.lower() in section.lower():
            return section
    
    return None

def extract_section(query):
    # Attempt to extract a specified section from the query
    query_tokens = query.lower().split()
    section_keywords = ["of", "about", "regarding", "related", "on"]
    for keyword in section_keywords:
        if keyword in query_tokens:
            index = query_tokens.index(keyword)
            if index < len(query_tokens) - 1:
                return " ".join(query_tokens[index+1:])
    return None

question = input("Please ask question related to long_text: ")
answer = answer_question(long_text, question)
print("Answer is: " + answer)

Please ask question related to long_text: Summary of Real Estate
Answer is: Real Estate: So far, three companies within our Coverage Universe have declared their results – Macrotech, Mahindra Lifespace, and Godrej Properties. All three companies have a strong pipeline for FY25 as well and are targeting to sustain the momentum going forward. Additionally, all companies within our Coverage Universe (barring DLF) have reported their operational performance and the cumulative bookings for 4QFY24 stood at INR264b, up 63% YoY.
